# Pilot på valideringsutvalget: designvalgene for RAG-pipelinen og agenten

Designvalgene testes her på et **eget valideringsutvalg**, slik at de bygger på andre spørsmål enn de 50 dev-spørsmålene promptene ble utviklet på: 100 spørsmål fra FinQAs treningssett (`data/utvalg_val.csv`, seed 42), med samme filter som de andre utvalgene. Ingen modeller trenes på treningssettet, og ingen av disse spørsmålene ble brukt under utviklingen.

Modell (`gpt-4.1`, temperatur 0), korrekthetsregel (relativ feil < 1 %) og kode er de samme som på dev og i testkjøringen. Testutvalget er urørt.

| Valg | Valgt oppsett | Alternativ | Gjelder | Hvorfor alternativet er rimelig (forel. 4) |
|---|---|---|---|---|
| Søkemetode | Dense (all-MiniLM-L6-v2) | Sparse (TF-IDF) | RAG-pipelinen og agenten (`src/korpus.py`) | Sparse treffer presist på tall og år |
| Chunking | Én setning per chunk | Grupper på 3 setninger | RAG-pipelinen og agenten (`src/korpus.py`) | Større chunks bevarer mer sammenheng |
| Antall treff | `TOP_K = 5` | 1 og 3 | RAG-pipelinen og hvert agentsøk | Flere chunks gir mer dekning, men mer støy |
| Prompt | Med regel mot nøsting | Uten regelen | Bare RAG-pipelinen | Færre regler gir en enklere prompt |

**Fremgangsmåte.** Som i øvingen endres **én innstilling om gangen**, og alt annet holdes likt med det valgte oppsettet. Valgene testes med RAG-pipelinen, fordi RAG-pipelinen gjør nøyaktig ett søk og ett modellkall per spørsmål. Da skyldes en forskjell innstillingen og ikke agentens egne valg.

1. **Retrieval alene** (ingen LLM, gratis): hvor ofte er alle gold-chunks hentet, og hvor mye tekst får modellen?
2. **Hele RAG-pipelinen** for det valgte oppsettet og hvert alternativ (100 API-kall per oppsett, 600 i alt).
3. **Parvis sammenligning** med McNemars eksakte test: alle oppsettene svarer på de samme spørsmålene, så vi teller spørsmålene der bare det ene oppsettet ble riktig.

Med 100 spørsmål er intervallene smalere enn på dev, men små forskjeller er fortsatt usikre.

## 0. Oppsett

In [1]:
# --- Les inn endringer i src/ automatisk ---
# Uten dette husker Python den gamle versjonen av en modul selv om filen er endret,
# og kjernen må startes på nytt for at endringen skal virke.
%load_ext autoreload
%autoreload 2

import re          # regulære uttrykk: finner programlinjen i modellens svar
import sys         # gir tilgang til Pythons søkesti, så vi kan importere fra src/
import time        # stoppeklokke: tidsbruk per spørsmål logges
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest     # eksakt McNemar-test: binomialtest på de uenige parene

# --- Finn prosjektroten og gjør src/ importerbar ---
# Notebooken ligger i notebooks/, så prosjektroten er ett nivå opp.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# --- Felles innstillinger og byggeklosser (samme som RAG-pipelinen og agenten) ---
# Piloten bruker nøyaktig den samme koden som de to systemene. Da gjelder det vi finner
# her, også for dem.
from src.config import MODEL, TOP_K, RESULTS_DIR, settings          # valgt oppsett (modell, top_k osv.)
from src.utvalg import load_utvalg, load_examples                   # spørsmålene og rapportsidene
from src.korpus import Korpus, gold_coverage                        # chunking, søk og dekningsmål
from src.kalkulator import run_program                              # kalkulatoren som regner ut programmet
from src.prompts import B2_SYSTEM_PROMPT, B2_EXAMPLE_USER, B2_EXAMPLE_ASSISTANT   # RAG-pipelinens prompt
from src.llm import call_model, usage                               # modellkall og tokentelling
from src.evaluering import is_correct, is_correct_rounded, classify_failure       # felles retting
from src.logg import make_record, write_jsonl, read_jsonl, same_settings          # felles loggformat

# --- Valideringsutvalget (fra train.json). Ikke bytt til test! ---
# Designvalgene testes på andre spørsmål enn testutvalget. Ellers ville vi tilpasset
# oppsettet til de samme spørsmålene som brukes i den endelige sammenligningen.
SPLIT = "val"
CONDITION = "RAG-pipeline"     # piloten bruker RAG-pipelinen (se forklaringen over)

pd.set_option("display.max_colwidth", 120)     # vis lengre tekst i tabellene før den kuttes
# Siste linje vises som utdata: oppsettet piloten tar utgangspunkt i.
{"condition": CONDITION, "split": SPLIT, **settings()}

{'condition': 'RAG-pipeline',
 'split': 'val',
 'model': 'gpt-4.1',
 'temperature': 0,
 'seed': 42,
 'retrieval_scope': 'dokument',
 'chunk_strategy': 'setning',
 'index_backend': 'embedding',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'top_k': 5,
 'max_agent_steps': 8,
 'rel_tol': 0.01}

## 1. Utvalg og de tre korpusene

De 100 spørsmålene i valideringsutvalget (`data/utvalg_val.csv`). Det bygges tre korpus over de samme rapportsidene, der bare én innstilling skiller dem fra det valgte oppsettet:

- `korpus`: det valgte oppsettet (dense, én setning per chunk)
- `korpus_tfidf`: sparse retrieval med TF-IDF
- `korpus_gruppe`: grupper på 3 setninger per chunk (`SENTENCES_PER_CHUNK`)

In [2]:
# --- Steg 1: Les utvalget ---
# utvalg: én rad per spørsmål, med spørsmål, fasit (exe_ans), fasitprogram og gold_inds.
# examples: oppslag fra spørsmålets id til hele rapportsiden (pre_text, tabell, post_text).
utvalg = load_utvalg(SPLIT)
examples = load_examples(SPLIT)
sider = {qid: examples[qid] for qid in utvalg["id"]}     # bare rapportsidene i utvalget

# --- Steg 2: Bygg de tre korpusene (indekseringstid, forel. 4) ---
# Hvert korpus deler rapportsidene i chunks og lager en søkeindeks én gang.
# De to alternativene endrer bare én innstilling hver, resten er likt det valgte oppsettet.
korpus = Korpus(sider)                                    # valgt oppsett, som i src/config.py
korpus_tfidf = Korpus(sider, backend="tfidf")             # endrer bare søkemetoden
korpus_gruppe = Korpus(sider, chunk_strategy="gruppe")    # endrer bare chunkingen

# --- Steg 3: Kontroll: hvor mange chunks ble det? ---
# Gruppering på 3 setninger gir færre, men lengre chunks enn én setning per chunk.
print({"spørsmål": len(utvalg),
       "chunks (setning)": sum(len(c) for c in korpus.chunks.values()),
       "chunks (gruppe)": sum(len(c) for c in korpus_gruppe.chunks.values())})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'spørsmål': 100, 'chunks (setning)': 2792, 'chunks (gruppe)': 1441}


## 2. RAG-pipelinen

Funksjonene under er de samme som i `RAG-pipeline.ipynb` (celle 5–8): kontekstpakking, ett modellkall som gir ett program, beregning med kalkulatoren og retting. Den eneste forskjellen er at `run_b2` her tar imot et annet korpus eller en annen systemprompt, og logger hvilket oppsett raden kom fra i `settings`.

In [3]:
MAX_CONTEXT_CHARS = 4000   # som i RAG-pipelinen: øvre grense for konteksten (omtrent 1000 tokens)

# --- Steg 1: Kontekstpakking (som i RAG-pipelinen, celle 5) ---
# Setter de hentede chunkene sammen til én tekst som sendes til modellen.
# Returnerer konteksten og chunkene som faktisk kom med.
def pack_context(hits: list, max_chars: int = MAX_CONTEXT_CHARS) -> tuple[str, list]:
    blocks, used_hits, used = [], [], 0
    for h in hits:                                      # rangert rekkefølge: beste treff først
        # Chunk-ID og type foran teksten, så modellen ser hvor tallene kommer fra.
        block = f"[{h['chunk_id']} | {h['kind']}]\n{h['text']}"
        if blocks and used + len(block) > max_chars:    # stopp før grensen, men ta alltid med minst én
            break
        blocks.append(block)
        used_hits.append(h)
        used += len(block)
    return "\n\n".join(blocks), used_hits             # blokkene skilles med en tom linje


# --- Steg 2: Input og programuttrekk (som i RAG-pipelinen, celle 6) ---
# Meldingene til modellen: ett eksempel (spørsmål + riktig program), så det ekte spørsmålet.
# Eksempelet viser modellen formatet vi vil ha svaret i.
def build_input(question: str, context: str) -> list:
    return [
        {"role": "user", "content": B2_EXAMPLE_USER},
        {"role": "assistant", "content": B2_EXAMPLE_ASSISTANT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


# Mønster for en programlinje: en av de fire operasjonene etterfulgt av en parentes.
PROGRAM_LINE_RE = re.compile(r"(?:add|subtract|multiply|divide)\(")

# Plukker ut programmet fra modellens svar. Modellen skriver av og til mer enn programmet
# (f.eks. "Program: ..." eller kodeblokk), så vi tar første linje som ser ut som et program.
def extract_program(raw: str) -> str:
    for line in (raw or "").strip().strip("`").splitlines():
        if PROGRAM_LINE_RE.search(line):
            # "Program: add(1, 2)" -> "add(1, 2)"
            return line.split(":", 1)[-1].strip() if line.lower().startswith("program") else line.strip()
    return (raw or "").strip()                          # ingen programlinje: send hele svaret videre


# --- Steg 3: ETT kall til LLM-en, ingen retry. instructions kan byttes ut i eksperimentet. ---
# I eksperimentet med regelen mot nøsting sendes en annen systemprompt inn her.
def generate_program(question: str, context: str, instructions: str = B2_SYSTEM_PROMPT) -> dict:
    response = call_model(build_input(question, context), instructions=instructions)
    raw = response.output_text
    # Returnerer råsvaret, programmet og tokenbruken (for kostnadsmålet).
    return {"raw_output": raw, "generated_expression": extract_program(raw), **usage(response)}


# --- Steg 4: Beregning (som i RAG-pipelinen, celle 7) ---
# Kjører programmet i den felles kalkulatoren. Et ugyldig program gir en feilmelding
# i stedet for at hele kjøringen stopper.
def compute(program: str) -> tuple[float | None, str | None]:
    try:
        return run_program(program), None               # (svar, ingen feil)
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"     # (ingen svar, feilmelding)


# --- Steg 5: Hele kjeden for ett spørsmål (som i RAG-pipelinen, celle 8) ---
# Søk -> kontekst -> ett modellkall -> kalkulator -> retting. Gir én loggrad per spørsmål.
# top_k, corpus og instructions er innstillingene eksperimentet kan endre.
# variant: innstillingene som skiller seg fra src/config.py. De logges i settings.
def run_b2(q: pd.Series, top_k: int = TOP_K, corpus=None, instructions: str = B2_SYSTEM_PROMPT,
           variant: dict | None = None) -> dict:
    start = time.time()
    gold = float(q["gold_answer"])
    # Feltene som er kjent før kjeden starter. De kommer med i loggraden uansett, også ved API-feil.
    base = dict(condition=CONDITION, split=SPLIT, id=q["id"], question=q["question"], gold_answer=gold,
                gold_program=q["gold_program"], gold_inds=list(q["gold_inds"]), n_steps=int(q["n_steps"]),
                model=MODEL, settings={**settings(), "top_k": top_k, **(variant or {})})
    # Søk i spørsmålets egen rapportside (valgt korpus hvis ikke et annet er gitt).
    hits = (corpus or korpus).search(q["question"], doc_id=q["id"], k=top_k)
    context, packed = pack_context(hits)
    # Fikk modellen se evidensen fasiten bygger på? Måles på det som faktisk ble sendt.
    all_gold, any_gold = gold_coverage(packed, q["gold_inds"])
    try:
        gen = generate_program(q["question"], context, instructions)
    except Exception as exc:                            # API-feil er ikke feil i RAG-pipelinen
        return make_record(**base, error=f"{type(exc).__name__}: {exc}", seconds=round(time.time() - start, 2))
    output, calc_error = compute(gen["generated_expression"])
    correct = is_correct(output, gold)                  # hovedregel: relativ feil under 1 %
    # Én loggrad med de avtalte feltene (src/logg.py), samme format som RAG-pipelinen og agenten.
    return make_record(
        **base,
        answer=output, raw_answer=gen["raw_output"],
        tool_calls=[{"name": "run_program", "arguments": gen["generated_expression"],
                     "output": calc_error if calc_error else output}],
        retrieved_chunks=[{"chunk_id": h["chunk_id"], "source_ids": h["source_ids"], "text": h["text"]} for h in packed],
        n_model_calls=1, n_tool_calls=1, stopped="svar",
        input_tokens=gen["input_tokens"], output_tokens=gen["output_tokens"],
        reasoning_tokens=gen["reasoning_tokens"],
        seconds=round(time.time() - start, 2), error=None,
        correct=correct, correct_rounded=is_correct_rounded(output, gold),
        gold_support_retrieved=all_gold, any_gold_retrieved=any_gold,
        failure_stage=classify_failure(correct, output, gold, all_gold),   # hvor i kjeden det gikk galt
    )

## 3. Retrieval alene (ingen LLM)

For hvert korpus søker vi én gang per spørsmål, med spørsmålet som query, slik RAG-pipelinen gjør. Målene er de samme som i RAG-pipeline-notebooken: **alle** gold-chunks hentet, **minst én** hentet, og hvor mange tegn konteksten blir.

Tegnene er viktige for chunkingen: fem grupper på tre setninger gir omtrent dobbelt så mye tekst som fem setninger. For å sammenligne på lik tekstmengde viser vi også dekningen for `top_k` = 1–10 med det valgte korpuset.

In [4]:
# Denne cellen bruker ingen API-kall: den måler bare søket.
# Mål: hvor ofte havner evidensen fasiten bygger på (gold_inds) i konteksten?
#   alle_gold_hentet:     ALLE gold-chunks kom med (nødvendig for å kunne svare riktig)
#   minst_en_gold_hentet: minst én gold-chunk kom med
#   tegn_median:          hvor lang konteksten ble (mer tekst = flere tokens = høyere kostnad)

# --- Del 1: Søkemetode og chunking med top_k = 5 ---
rader = []
# Steg 1: Søk med hvert av de tre korpusene, ett søk per spørsmål (slik RAG-pipelinen gjør).
for navn, kp in [("valgt oppsett", korpus), ("sparse (TF-IDF)", korpus_tfidf),
                 ("grupper på 3 setninger", korpus_gruppe)]:
    for _, q in utvalg.iterrows():
        ctx, packed = pack_context(kp.search(q["question"], doc_id=q["id"]))
        alle, noen = gold_coverage(packed, q["gold_inds"])
        rader.append({"oppsett": navn, "alle_gold_hentet": alle, "minst_en_gold_hentet": noen, "tegn": len(ctx)})
# Steg 2: Snitt per oppsett. Gjennomsnittet av True/False er andelen spørsmål.
dekning = pd.DataFrame(rader).groupby("oppsett", sort=False).agg(
    alle_gold_hentet=("alle_gold_hentet", "mean"), minst_en_gold_hentet=("minst_en_gold_hentet", "mean"),
    tegn_median=("tegn", "median"))
print("--- søkemetode og chunking, top_k = 5 ---")
# Steg 3: Vis andelene som prosent.
display(dekning.assign(alle_gold_hentet=dekning.alle_gold_hentet.map("{:.0%}".format),
                       minst_en_gold_hentet=dekning.minst_en_gold_hentet.map("{:.0%}".format)))

# --- Del 2: top_k = 1-10 med det valgte korpuset, med tekstmengde ---
# Rangeringen endres ikke med k, så vi henter de 10 beste én gang og ser på de k første.
# Flere treff gir høyere dekning, men lengre kontekst. Tabellen viser avveiningen.
rader = []
for _, q in utvalg.iterrows():
    top10 = korpus.search(q["question"], doc_id=q["id"], k=10)
    for k in range(1, 11):
        ctx, packed = pack_context(top10[:k])           # de k beste treffene
        alle, noen = gold_coverage(packed, q["gold_inds"])
        rader.append({"top_k": k, "alle_gold_hentet": alle, "minst_en_gold_hentet": noen, "tegn": len(ctx)})
# Snitt per top_k, samme mål som i del 1.
topk_dekning = pd.DataFrame(rader).groupby("top_k").agg(
    alle_gold_hentet=("alle_gold_hentet", "mean"), minst_en_gold_hentet=("minst_en_gold_hentet", "mean"),
    tegn_median=("tegn", "median"))
print("--- top_k, valgt korpus ---")
topk_dekning.assign(alle_gold_hentet=topk_dekning.alle_gold_hentet.map("{:.0%}".format),
                    minst_en_gold_hentet=topk_dekning.minst_en_gold_hentet.map("{:.0%}".format))

--- søkemetode og chunking, top_k = 5 ---


,alle_gold_hentet,minst_en_gold_hentet,tegn_median
oppsett,,,
valgt oppsett,64%,90%,821.5
sparse (TF-IDF),52%,83%,884.5
grupper på 3 setninger,68%,92%,1446.0


--- top_k, valgt korpus ---


,alle_gold_hentet,minst_en_gold_hentet,tegn_median
top_k,,,
1,22%,50%,149.0
2,35%,67%,315.0
3,50%,81%,467.0
4,59%,86%,660.5
5,64%,90%,821.5
6,69%,91%,1011.5
7,69%,92%,1139.0
8,73%,93%,1265.0
9,75%,95%,1459.5


## 4. Hele RAG-pipelinen

Det valgte oppsettet og hvert alternativ kjøres på alle 100 spørsmålene, også `top_k` = 1 og 3. Det finnes ingen tidligere kjøringer på valideringsutvalget, så alt kjøres her.

**Ingen dobbeltbetaling:** alle radene lagres i `results/rag_pipeline_val_designvalg.jsonl` og gjenbrukes hvis oppsettet i `src/config.py` er uendret.

**Regelen mot nøsting** fjernes ved å ta ut de to linjene fra RAG-pipeline-prompten. Resten av prompten er ordrett lik. Et program regnes som nøstet hvis en operasjon står inni parentesen til en annen, for eksempel `divide(a, add(b, c))`.

In [5]:
# --- Steg 1: Prompten uten regelen mot nøsting ---
# Regelen ber modellen skrive ett regnesteg om gangen (#0, #1, ...) i stedet for å legge
# operasjoner inni hverandre. Vi lager en kopi av systemprompten der bare denne regelen er fjernet.
NESTING_RULE = ("- Never nest operations inside each other. Use #0, #1, ... instead:\n"
                "  write add(60.1, 161.9), divide(161.9, #0), not divide(161.9, add(60.1, 161.9)).\n")
assert NESTING_RULE in B2_SYSTEM_PROMPT          # stopper hvis prompten er endret i src/prompts.py
PROMPT_UTEN_NOSTING = B2_SYSTEM_PROMPT.replace(NESTING_RULE, "")

# --- Steg 2: Alternativene, hvert med én endret innstilling ---
# Nøkkel: navnet i tabellene. Verdi: argumentene til run_b2 som avviker fra valgt oppsett.
# variant logges i settings, slik at hver rad i resultatfilen viser hvilket oppsett den kom fra.
ALTERNATIVER = {
    "valgt oppsett":          dict(),                                         # sammenligningsgrunnlaget
    "top_k = 1":              dict(top_k=1),
    "top_k = 3":              dict(top_k=3),
    "sparse (TF-IDF)":        dict(corpus=korpus_tfidf, variant={"index_backend": "tfidf"}),
    "grupper på 3 setninger": dict(corpus=korpus_gruppe, variant={"chunk_strategy": "gruppe"}),
    "uten regel mot nøsting": dict(instructions=PROMPT_UTEN_NOSTING, variant={"prompt": "uten_nosteregel"}),
}

# --- Steg 3: Kjør, eller les inn en lagret kjøring med samme oppsett ---
RERUN = False                                             # True = kjør på nytt (6 x 100 API-kall)
DESIGN_PATH = RESULTS_DIR / f"rag_pipeline_{SPLIT}_designvalg.jsonl"
VARIERER = ("index_backend", "chunk_strategy", "prompt", "top_k")   # innstillingene som får variere

# Den lagrede filen brukes bare hvis resten av oppsettet (modell, temperatur osv.) er uendret.
lagret = read_jsonl(DESIGN_PATH) if DESIGN_PATH.exists() else None
if lagret is not None and not RERUN and same_settings(lagret, settings(), ignore=VARIERER):
    # Steg 3a: Les inn den lagrede kjøringen (ingen API-kall).
    design = lagret
    print(f"Leste {len(design)} rader fra {DESIGN_PATH.relative_to(ROOT)} (sett RERUN = True for å kjøre på nytt)")
else:
    # Steg 3b: Kjør hele kjeden for hvert alternativ på alle 100 spørsmålene, og lagre resultatet.
    rader, start = [], time.time()
    for navn, args in ALTERNATIVER.items():
        rader += [run_b2(q, **args) for _, q in utvalg.iterrows()]
        print(f"{navn} ferdig, {time.time() - start:.0f} s")
    write_jsonl(rader, DESIGN_PATH)
    design = pd.DataFrame(rader)

# --- Steg 4: Skill ut API-feil (ikke feil i RAG-pipelinen) ---
# Rader med API-feil har ikke noe svar og holdes utenfor sammenligningen.
alle = design[design["error"].isna()].copy()
print({"rader": len(design), "API-feil": int(design["error"].notna().sum())})


# --- Steg 5: Merk hver rad med oppsettets navn ut fra settings ---
# Resultatfilen lagrer bare innstillingene. Her gjøres de om til navnene fra ALTERNATIVER.
def oppsett_navn(s: dict) -> str:
    if s.get("prompt") == "uten_nosteregel":
        return "uten regel mot nøsting"
    if s["index_backend"] == "tfidf":
        return "sparse (TF-IDF)"
    if s["chunk_strategy"] == "gruppe":
        return "grupper på 3 setninger"
    if s["top_k"] != TOP_K:
        return f"top_k = {s['top_k']}"
    return "valgt oppsett"

alle["oppsett"] = alle["settings"].map(oppsett_navn)
# Programmet modellen skrev (argumentet til kalkulatoren).
alle["program"] = alle["tool_calls"].map(lambda calls: calls[0]["arguments"] if calls else "")
# Nøstet program: en parentes inni en annen parentes, f.eks. divide(1, add(2, 3)).
alle["nostet"] = alle["program"].str.contains(r"\([^()]*\(", regex=True)

# --- Steg 6: Oppsummering per oppsett ---
# Antall riktige svar, hvor ofte all evidens ble hentet, kalkulatorfeil, nøstede program
# og gjennomsnittlig antall input-tokens (kostnad) for hvert oppsett.
REKKEFOLGE = ["valgt oppsett", "sparse (TF-IDF)", "grupper på 3 setninger", "top_k = 1", "top_k = 3",
              "uten regel mot nøsting"]
sammendrag = alle.groupby("oppsett").agg(
    spørsmål=("id", "size"), riktige=("correct", "sum"), alle_gold_hentet=("gold_support_retrieved", "sum"),
    kalkulatorfeil=("failure_stage", lambda s: int((s == "kalkulator").sum())),
    nostede_program=("nostet", "sum"), input_tokens_snitt=("input_tokens", "mean")).round(0).astype(int)
sammendrag.loc[REKKEFOLGE]                                # vis radene i fast rekkefølge

valgt oppsett ferdig, 115 s


top_k = 1 ferdig, 230 s


top_k = 3 ferdig, 341 s


sparse (TF-IDF) ferdig, 572 s


grupper på 3 setninger ferdig, 683 s


uten regel mot nøsting ferdig, 795 s
{'rader': 600, 'API-feil': 0}


,spørsmål,riktige,alle_gold_hentet,kalkulatorfeil,nostede_program,input_tokens_snitt
oppsett,,,,,,
valgt oppsett,100,60,64,5,1,631
sparse (TF-IDF),100,50,52,11,0,626
grupper på 3 setninger,100,60,68,4,0,767
top_k = 1,100,25,22,31,2,433
top_k = 3,100,48,50,6,0,538
uten regel mot nøsting,100,59,64,6,2,572


## 5. Parvis sammenligning mot det valgte oppsettet

For hvert alternativ teller vi spørsmålene der **bare** alternativet ble riktig, og der **bare** det valgte oppsettet ble riktig. Spørsmål der begge er riktige eller begge feil, sier ingenting om forskjellen.

**McNemars eksakte test:** hvis oppsettene er like gode, er hvert uenige spørsmål som et myntkast. p-verdien er sannsynligheten for en minst like skjev fordeling under denne antakelsen. Samme test brukes for å sammenligne RAG-pipelinen og agenten på testsettet.

In [6]:
# --- Steg 1: Én kolonne per oppsett, én rad per spørsmål ---
# Verdien er True hvis oppsettet svarte riktig på spørsmålet. Da kan vi sammenligne
# to oppsett på nøyaktig de samme spørsmålene (parvis sammenligning).
bred = alle.pivot_table(index="id", columns="oppsett", values="correct", aggfunc="first").astype(bool)

# --- Steg 2: Uenige par og eksakt McNemar-test for hvert alternativ ---
# Spørsmål der begge oppsettene har rett (eller begge feil), sier ingenting om hvilket som er best.
# Testen ser derfor bare på de uenige spørsmålene: der bare det ene oppsettet har rett.
parvis = []
for navn in REKKEFOLGE[1:]:                               # alle alternativene, ikke valgt oppsett selv
    bare_alt = int((bred[navn] & ~bred["valgt oppsett"]).sum())     # alternativet rett, valgt feil
    bare_valgt = int((~bred[navn] & bred["valgt oppsett"]).sum())   # valgt rett, alternativet feil
    n = bare_alt + bare_valgt                              # antall uenige spørsmål
    parvis.append({"alternativ": navn, "riktige (alternativ)": int(bred[navn].sum()),
                   "riktige (valgt)": int(bred["valgt oppsett"].sum()),
                   "bare alternativet riktig": bare_alt, "bare valgt oppsett riktig": bare_valgt,
                   # Er oppsettene like gode, er hvert uenige spørsmål et myntkast (p = 0.5).
                   # Ingen uenige spørsmål: ingen forskjell å teste, p settes til 1.
                   "p (McNemar, eksakt)": round(binomtest(bare_alt, n, 0.5).pvalue, 3) if n else 1.0})
pd.DataFrame(parvis)

,alternativ,riktige (alternativ),riktige (valgt),bare alternativet riktig,bare valgt oppsett riktig,"p (McNemar, eksakt)"
0,sparse (TF-IDF),50,60,5,15,0.041
1,grupper på 3 setninger,60,60,4,4,1.000
2,top_k = 1,25,60,2,37,0.000
3,top_k = 3,48,60,0,12,0.000
4,uten regel mot nøsting,59,60,1,2,1.000


## 6. Konklusjon: begrunnelse for valgene

| Valg | Resultat på valideringsutvalget (100 spørsmål) | Beslutning |
|---|---|---|
| **Dense mot TF-IDF** | Alle gold hentet for 64 % mot 52 %. 60 mot 50 riktige (15 mot 5 i de uenige parene, p = 0,04). Færre kalkulatorfeil (5 mot 11). | **Dense beholdes.** |
| **Én setning mot grupper på 3** | Dekning 64 % mot 68 %, men grupper gir nesten dobbelt så mye tekst. 60 mot 60 riktige (4 mot 4, p = 1,0). Med lik tekstmengde gir setninger høyere dekning: 9 setninger (~1460 tegn) gir 75 %, mot 68 % for 5 grupper (~1450 tegn). | **Setning beholdes.** |
| **`TOP_K = 5` mot 3 og 1** | Dekning 64 % mot 50 % og 22 %. 60 mot 48 riktige (12 mot 0, p < 0,001) og 60 mot 25 (37 mot 2, p < 0,001). Over k = 5 øker dekningen saktere (77 % med k = 10). | **k = 5 beholdes.** |
| **Med mot uten regel mot nøsting** | 60 mot 59 riktige (2 mot 1, p = 1,0). Modellen nøstet nesten aldri (1 mot 2 nøstede program). | **Regelen beholdes** som sikring uten kostnad. |

Forskjellene for søkemetode og antall treff er statistisk signifikante. Chunking og regelen mot nøsting gir ingen målbar forskjell, og der beholdes det enkleste og billigste oppsettet.